<a href="https://colab.research.google.com/github/naylaakas/PembelajaranMesin-2026/blob/main/JS06/datasetPBL/Praktikum6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install annoy faiss-cpu hnswlib

In [5]:
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# -------------------------------
# Load dataset milikmu (cafes.csv & reviews.csv)
# -------------------------------
df_cafes = pd.read_csv('cafes.csv')
df_reviews = pd.read_csv('reviews.csv')
df = pd.merge(df_reviews, df_cafes, on='cafe_id', how='left', suffixes=('_review', '_cafe'))

# Pilih fitur numerik saja
features = ['rating_review', 'rating_cafe', 'review_count', 'latitude', 'longitude']
X = df[features].fillna(0).values

# Standarisasi fitur menggunakan StandardScaler (float32 untuk FAISS & HNSW)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

k = 10  # jumlah nearest neighbors

# -------------------------------
# Exact Nearest Neighbor (brute-force)
# -------------------------------
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.3f} s")

# -------------------------------
# Annoy
# -------------------------------
start = time.time()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10)
idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled]
time_annoy = time.time() - start
print(f"Annoy done in {time_annoy:.3f} s")

# -------------------------------
# HNSW
# -------------------------------
start = time.time()
p_hnsw = hnswlib.Index(space='l2', dim=X_scaled.shape[1])
p_hnsw.init_index(max_elements=X_scaled.shape[0], ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
idx_hnsw, dist_hnsw = p_hnsw.knn_query(X_scaled, k=k)
time_hnsw = time.time() - start
print(f"HNSW done in {time_hnsw:.3f} s")

# -------------------------------
# FAISS IVF (perbaikan: gunakan positional argument tanpa nlist= dan metric=)
# -------------------------------
start = time.time()
quantizer = faiss.IndexFlatL2(X_scaled.shape[1])
index_faiss = faiss.IndexIVFFlat(quantizer, X_scaled.shape[1], 50, faiss.METRIC_L2)
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 10
dist_faiss, idx_faiss = index_faiss.search(X_scaled, k)
time_faiss = time.time() - start
print(f"FAISS IVF done in {time_faiss:.3f} s")

# -------------------------------
# Contoh tampilkan top-5 neighbors dari item pertama
# -------------------------------
print(f"\nItem pertama (Index 0): Kafe '{df.loc[0, 'name']}' | Rating Review: {df.loc[0, 'rating_review']} | Rating Kafe: {df.loc[0, 'rating_cafe']}")
print("\nTop-5 neighbors for first review/cafe item:")
print(f"Exact NN : {idx_exact[0][:5]}")
print(f"Annoy    : {idx_annoy[0][:5]}")
print(f"HNSW     : {idx_hnsw[0][:5]}")
print(f"FAISS    : {idx_faiss[0][:5]}")

# Tampilkan detail nama kafe dari Top-5 neighbors (Exact NN vs ANN)
print("\nDetail Top-5 Kafe Terdekat (berdasarkan Exact NN / HNSW):")
print(df.loc[idx_exact[0][:5], ['cafe_id', 'name', 'rating_review', 'rating_cafe', 'review_count']])

Exact NN done in 0.043 s
Annoy done in 0.109 s
HNSW done in 0.530 s
FAISS IVF done in 0.050 s

Item pertama (Index 0): Kafe 'ARTE COFFEE & EATERY' | Rating Review: 5.0 | Rating Kafe: 4.7

Top-5 neighbors for first review/cafe item:
Exact NN : [ 3 11  5  9  2]
Annoy    : [0, 1, 2, 3, 4]
HNSW     : [ 4  8 13 15 18]
FAISS    : [0 1 2 3 4]

Detail Top-5 Kafe Terdekat (berdasarkan Exact NN / HNSW):
                                 cafe_id                  name  rating_review  \
3   ba4b512c-fa2d-4959-8384-8570c81121ec  ARTE COFFEE & EATERY            5.0   
11  ba4b512c-fa2d-4959-8384-8570c81121ec  ARTE COFFEE & EATERY            5.0   
5   ba4b512c-fa2d-4959-8384-8570c81121ec  ARTE COFFEE & EATERY            5.0   
9   ba4b512c-fa2d-4959-8384-8570c81121ec  ARTE COFFEE & EATERY            5.0   
2   ba4b512c-fa2d-4959-8384-8570c81121ec  ARTE COFFEE & EATERY            5.0   

    rating_cafe  review_count  
3           4.7           528  
11          4.7           528  
5           4.7     